Notebook outline:

Setup & paths

Import loaders (including your load_trim_data.py)

Load core tables (Basic, Price, Sales)

Load + merge Trim table parts via your script

Harmonize column names and keys

Build a master DataFrame (model‑year × trim)

Quick EDA checks

Minimal feature engineering pass (complexity proxies)

Save processed datasets (Parquet)

In [ ]:
# This notebook:
# 1) Loads DVM-CAR tables (local CSVs split for GitHub)
# 2) Uses your load_trim_data.py to merge Trim_table parts
# 3) Builds a master dataset at model-year-trim level
# 4) Adds first-pass 'cost & complexity' proxy features
# 5) Saves processed Parquet files for downstream modeling

# %%
from pathlib import Path

# Project-relative paths
REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data" / "dvmcar"
PROC_DIR = REPO_ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

REPO_ROOT, DATA_DIR, PROC_DIR

In [ ]:
import sys
sys.path.append(str(REPO_ROOT))  # so we can import from src/ and project root

import pandas as pd
import numpy as np

# Import your loader for Trim parts
# Expecting a function like: load_trim_table(parts_dir: str) -> pd.DataFrame
from load_trim_data import load_trim_table

# (Optional) If you have additional utils, you can import them here later

In [ ]:
# %%
# Load the smaller tables you already committed to the repo
basic_path = DATA_DIR / "Basic_table.csv"
price_path = DATA_DIR / "Price_table.csv"
sales_path = DATA_DIR / "Sales_table.csv"  # optional, may not exist

basic = pd.read_csv(basic_path)
price = pd.read_csv(price_path)
sales = pd.read_csv(sales_path) if sales_path.exists() else None

basic.shape, price.shape, (sales.shape if sales is not None else None)

In [ ]:
# %%
# This will load e.g., Trim_table_part1.csv and Trim_table_part2.csv
trim = load_trim_table(str(DATA_DIR))
trim.shape

In [ ]:
# %%
def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = out.columns.str.strip().str.replace(" ", "_").str.replace("-", "_")
    return out

basic = normalize_cols(basic)
price = normalize_cols(price)
trim  = normalize_cols(trim)
if sales is not None:
    sales = normalize_cols(sales)

basic.columns.tolist()[:20], price.columns.tolist()[:20], trim.columns.tolist()[:20]

In [ ]:
# %%
# Try to standardize common columns across tables.
# Adjust mappings below if your actual CSV headers differ.

# For 'basic' table
for cand in ["Automaker", "Maker"]:
    if cand in basic.columns:
        basic.rename(columns={cand: "Automaker"}, inplace=True)
        break
if "Genmodel_id" in basic.columns:
    basic.rename(columns={"Genmodel_id": "Genmodel_ID"}, inplace=True)

# For 'price' table
if "Entry_Price" in price.columns and "Entry_price" not in price.columns:
    price.rename(columns={"Entry_Price": "Entry_price"}, inplace=True)
if "Genmodel_id" in price.columns:
    price.rename(columns={"Genmodel_id": "Genmodel_ID"}, inplace=True)

# For 'trim' table — it should contain specs (engine type/size, fuel, trim name, etc.)
if "Genmodel_id" in trim.columns:
    trim.rename(columns={"Genmodel_id": "Genmodel_ID"}, inplace=True)
if "Trim" in trim.columns and "Trim_Name" not in trim.columns:
    trim.rename(columns={"Trim": "Trim_Name"}, inplace=True)

# For 'sales' (optional)
if sales is not None and "Genmodel_id" in sales.columns:
    sales.rename(columns={"Genmodel_id": "Genmodel_ID"}, inplace=True)

# Make sure Year is numeric if present
for df in [price, trim, sales] if sales is not None else [price, trim]:
    if "Year" in df.columns:
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

basic.head()

In [ ]:
# %%
# 1) Merge TRIM (many rows) with BASIC (one row per model)
master = trim.merge(
    basic[["Genmodel_ID", "Automaker", "Genmodel"]].drop_duplicates(),
    on="Genmodel_ID",
    how="left"
)

# 2) Merge in annual entry price (contextual price per model-year)
if {"Genmodel_ID", "Year"}.issubset(price.columns):
    master = master.merge(
        price[["Genmodel_ID", "Year", "Entry_price"]],
        on=["Genmodel_ID", "Year"],
        how="left"
    )

master.shape, master.sample(3)

In [ ]:
# %%
# Basic null/duplicate checks
print("Rows:", len(master))
print("Columns:", len(master.columns))
print("\nNull counts (top 15):")
print(master.isna().sum().sort_values(ascending=False).head(15))

# Quick object vs numeric overview
master.dtypes.value_counts()

In [ ]:
# %%
def first_pass_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Normalize column names that we may rely on
    # Try to infer typical spec columns; adjust to your CSV headers
    numeric_candidates = ["Engine_size", "Trim_Price", "Entry_price"]
    for col in numeric_candidates:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    # Engine complexity proxy:
    # More complex if hybrid/plug-in/electric in Engine_type string; scale with engine size
    if "Engine_type" in out.columns:
        engine_type = out["Engine_type"].astype(str).str.lower()
        powertrain_flag = engine_type.str.contains("hybrid|plug|electric", regex=True).astype(float)
        size_term = (out.get("Engine_size", pd.Series([0]*len(out))).fillna(0) / 3000.0)
        out["engine_complexity_score"] = (1 + 0.5*powertrain_flag) * (1.0 + size_term)
    else:
        out["engine_complexity_score"] = 1.0

    # Feature density proxy (if Trim_Price present; else fall back to Entry_price)
    base_price = out.get("Trim_Price")
    if base_price is None or base_price.isna().all():
        base_price = out.get("Entry_price")
    out["feature_density"] = np.log1p(base_price.fillna(base_price.median() if base_price is not None else 0))

    # Price band quartiles from Entry_price
    if "Entry_price" in out.columns:
        try:
            out["price_band"] = pd.qcut(out["Entry_price"], q=4, labels=False, duplicates="drop")
        except Exception:
            out["price_band"] = 0
    else:
        out["price_band"] = 0

    # A simple consolidated "design_complexity_index"
    out["design_complexity_index"] = (
        out["engine_complexity_score"].fillna(1.0) +
        0.5 * out["feature_density"].fillna(out["feature_density"].median())
    )

    return out

feat = first_pass_features(master)
feat[["engine_complexity_score","feature_density","price_band","design_complexity_index"]].describe()

In [ ]:
# %%
master_path = PROC_DIR / "vehicle_master.parquet"
feat_path   = PROC_DIR / "vehicle_master_features.parquet"

master.to_parquet(master_path, index=False)
feat.to_parquet(feat_path, index=False)

master_path, feat_path

In [ ]:
# %%
import seaborn as sns
import matplotlib.pyplot as plt

subset = feat[["engine_complexity_score","feature_density","design_complexity_index","Entry_price"]].copy()
subset = subset.dropna()
g = sns.pairplot(subset.sample(min(len(subset), 2000), random_state=42),
                 diag_kind="kde")
plt.suptitle("Quick sanity check — feature relationships", y=1.02)
plt.show()